# 文ごとの採点役の確認（M4-S・docs/v2-light-model-plan.md「承認後に固定する下書き」・Colab T4 用）

**本人の承認（「採点OK」）の後にだけ流す。承認前は流さない。**

やること: 文ごとに 0/1 を付ける採点役が、正解の分かっている 11 例（evaluation/v2/judge_precheck_s.jsonl・既存 5 + 別題材 6）を正しく採点できるかを確かめる。通った採点役だけを、保存済みの回答の採点し直しに使える。

決まり（実行前に固定。結果を見て変えない）:

- 試す採点役は既にある 2 つだけ。**ELYZA-JP 8B → Mistral-Nemo 12B** の順に試し、最初に通った物を使う。phi-4 の再試行・新しい採点役・有料 API はしない
- 同じ設定で 2 回採点し、11 例が 2 回とも通れば合格。どちらかの回に採点失敗（読めない出力・数が合わない・呼び出しの失敗）が 1 件でもあれば「未判定」で、その採点役は使わない
- 温度 0（`LLM_TEMPERATURE=0`）・出力の上限 `LLM_MAX_TOKENS=2048`。確認と採点し直しで同じ値にする（採点し直す側が温度・指示文・採点役の一致を確かめて、違えば止まる）
- 結果を見て指示文（evaluation/judges_sentence.py）や確認例を直さない。どちらも通らなければ人手採点（evaluation/human_sheet.py）へ
- 打ち切り: **2026-10-26 23:59 JST**。過ぎていたら準備セルで止まる

流れ（「Run all」で全部流れる）:

1. 準備（RAG 側の軽い仮想環境・vLLM の仮想環境）
2. 採点役を順に単独で起動 → 確認（`python -m evaluation.judge_precheck_s`）→ 結果の JSON を全部セルに出す → 最初に通った物で止める
3. （任意）通った採点役で、保存済みの回答（raw.jsonl）を採点し直す。`RAW_PATHS` が空なら飛ばす
4. 片付け・zip

結果は `/content/judge_s/` に残る。確認例は架空の社内規程なので、セルの出力に出してよい。

In [ ]:
# 依存の衝突を避けるため、vLLM と RAG 側を別の仮想環境に入れる。
# 採点役の確認は設定・OpenAI 互換の呼び出しだけを使うので、RAG 側は軽い依存だけでよい（埋め込み・Qdrant は使わない）
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git /content/repo
!uv venv -q /content/rag-env
!uv pip install -q --python /content/rag-env/bin/python pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm

In [ ]:
# 設定（実行前に固定。結果を見て変えない）
# 採点役: 上から順に試し、確認を最初に通った物を使う（10/23 の協議で決めた順）
JUDGE_CANDIDATES = [
    # ELYZA の日本語 8B（Meta Llama 3 系・Llama 3 Community License）。重み 5.7 GB・文脈 8192
    ("elyza/Llama-3-ELYZA-JP-8B-AWQ", "85c3670a70d1e53bb4faa3f01c3ac2878f4a3fcd"),
    # Mistral AI 12B（元は Apache-2.0・量子化版の札には明記なし）。重み 8.4 GB
    ("casperhansen/mistral-nemo-instruct-2407-awq", "c83b6438e13051ad1c0f5683635705ee83bb8772"),
]
JUDGE_EXTRA_ARGS = []
JUDGE_LEN, JUDGE_UTIL = 8192, 0.85       # 採点役は単独で起動する
JUDGE_TEMPERATURE = "0"                   # 確認と採点し直しで同じ値（0 以外だと両方とも実行を拒否する）
JUDGE_MAX_TOKENS = "2048"                 # 文ごとの根拠を書くので既定の 1024 より長く取る。確認と採点し直しで同じ値
CUTOFF_JST = "2026-10-26T23:59:00+09:00"  # 採点役探しの打ち切り
REPO = "/content/repo"
RAG_PY = "/content/rag-env/bin/python"
VLLM_BIN = "/content/vllm-env/bin/vllm"
OUT = "/content/judge_s"                  # 結果の置き場所（やり直すときはセルを流す前に消すか名前を替える）
PORT = 8001
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
# （任意）採点し直す保存済みの回答。Step 1 などの raw.jsonl を /content に上げてパスを並べる。空なら飛ばす
RAW_PATHS = []

In [ ]:
import datetime, json, os, pathlib, re, signal, subprocess, sys, time, urllib.request

if datetime.datetime.now(datetime.timezone.utc) > datetime.datetime.fromisoformat(CUTOFF_JST):
    raise RuntimeError(f"打ち切り（{CUTOFF_JST}）を過ぎた。採点役探しはしない → 人手採点へ")

sys.path.insert(0, REPO)   # evaluation.colab.step05 を使う（標準ライブラリだけなので kernel から import できる）
from evaluation.colab import step05 as s5

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）。消すか設定セルの OUT を替える")
OUTP.mkdir(parents=True, exist_ok=True)

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（落とすときに killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    # 起動したら True。プロセスが落ちた・時間切れなら、ログの抜き出しを出して False
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return True
        except Exception:
            time.sleep(3)
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|Exception|Killed", l)]
    print("--- 起動しない:", url, "（全文は", log, "）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "エラーらしい行なし")
    print("--- ログの末尾 30 行 ---")
    print("\n".join(lines[-30:]))
    return False

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def read_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return s5.parse_smi_mib(out)

GPU_NAME, CAP, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
VLLM_VER = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                          capture_output=True, text=True).stdout.strip() or "不明"
BASELINE_MIB = read_mib()   # 何も載せていない時の使用量（GPU が空いたかの基準）
JUDGE_REV_NOW = {m: s5.hf_revision(m) for m, _ in JUDGE_CANDIDATES}
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(GPU_NAME, "compute", CAP, "全体", TOTAL_MIB, "MiB / 起動前", BASELINE_MIB, "MiB / vllm", VLLM_VER, "/ commit", COMMIT)
print("revision（固定値と今の値）:", [(m, sha, JUDGE_REV_NOW[m].get("sha")) for m, sha in JUDGE_CANDIDATES])
JUDGE_ENV = dict(os.environ, **s5.eval_env({}, BASE_URL, LLM_TEMPERATURE=JUDGE_TEMPERATURE,
                                           LLM_MAX_TOKENS=JUDGE_MAX_TOKENS))

In [ ]:
# 2. 採点役を順に単独で起動 → 確認（11 例 × 2 回）→ 最初に通った物で止める（通った採点役は落とさず次のセルで使う）
JUDGE = {"role": "sentence-judge", "quant": "AWQ 4bit", "max_model_len": JUDGE_LEN, "gpu_util": JUDGE_UTIL,
         "temperature": JUDGE_TEMPERATURE, "max_tokens": JUDGE_MAX_TOKENS, "commit": COMMIT,
         "vllm_version": VLLM_VER, "gpu": GPU_NAME, "tried": []}
JUDGE_PROC, PRECHECK_PATH = None, None
for JUDGE_MODEL, JUDGE_SHA in JUDGE_CANDIDATES:
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    tag = JUDGE_MODEL.split("/")[-1]
    judge_log = OUTP / f"vllm_judge_{tag}.log"
    judge = launch(s5.vllm_argv(VLLM_BIN, JUDGE_MODEL, PORT, JUDGE_LEN, JUDGE_UTIL, JUDGE_SHA,
                                extra=JUDGE_EXTRA_ARGS), str(judge_log))
    trial = {"model": JUDGE_MODEL, "revision": JUDGE_SHA,
             "started": wait(f"{BASE_URL}/models", judge, str(judge_log), 1800)}
    JUDGE["tried"].append(trial)
    if not trial["started"]:
        trial["precheck"] = "未判定"
        print(f"[{tag}] 採点役が起動しない → 次の候補")
        kill(judge)
        continue
    pre_out = OUTP / f"judge_precheck_s_{tag}.json"
    p = subprocess.run([RAG_PY, "-m", "evaluation.judge_precheck_s", "--out", str(pre_out)],
                       cwd=REPO, env=JUDGE_ENV, capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-3000:])
    # 終了コード 0=pass / 1=fail / 3=未判定 / 2=前提違反（温度・fixture・--out）。JSON が無いときも未判定
    trial["precheck"] = json.loads(pre_out.read_text())["result"] if pre_out.exists() else "未判定"
    trial["precheck_rc"] = p.returncode
    if pre_out.exists():
        # 確認の全部（文ごとの判定・根拠・生の出力）をセルの出力に出す
        print(f"PRECHECK_S_{tag}_JSON " + json.dumps(json.loads(pre_out.read_text()), ensure_ascii=False))
    if trial["precheck"] != "pass":
        print(f"[{tag}] 確認が {trial['precheck']} → 次の候補")
        kill(judge)
        continue
    JUDGE.update({"model": JUDGE_MODEL, "revision": JUDGE_SHA, "precheck": "pass", "precheck_path": str(pre_out)})
    JUDGE_PROC, PRECHECK_PATH = judge, pre_out
    break
if JUDGE.get("precheck") != "pass":
    JUDGE["precheck"] = "fail" if any(t.get("precheck") == "fail" for t in JUDGE["tried"]) else "未判定"
    JUDGE["skipped"] = "どちらの採点役も確認を通らない → 人手採点へ（指示文・確認例は直さない）"
    print(JUDGE["skipped"])
print("JUDGE_S_JSON " + json.dumps(JUDGE, ensure_ascii=False))

In [ ]:
# 3.（任意）通った採点役で保存済みの回答を文ごとに採点し直す。RAW_PATHS が空・確認が通っていなければ飛ばす
REJUDGED = {}
if JUDGE.get("precheck") != "pass":
    print("確認が通っていないので採点し直さない")
elif not RAW_PATHS:
    print("RAW_PATHS が空なので飛ばす")
else:
    for i, raw in enumerate(RAW_PATHS):
        out = OUTP / f"rejudge_s_{i}_{pathlib.Path(raw).parent.name or 'raw'}.json"
        p = subprocess.run([RAG_PY, "-m", "evaluation.rejudge_s", "--raw", str(raw),
                            "--precheck", str(PRECHECK_PATH), "--out", str(out)],
                           cwd=REPO, env=JUDGE_ENV, capture_output=True, text=True)
        print(p.stdout[-3000:], p.stderr[-3000:])
        # 終了コード 0=すべて pass / 1=どれかが fail / 3=どれかが未判定 / 2=実行前の拒否
        REJUDGED[str(raw)] = {"rc": p.returncode, "out": str(out) if out.exists() else None}
        if out.exists():
            print(f"REJUDGE_S_{i}_JSON " + json.dumps(json.loads(out.read_text()), ensure_ascii=False))
print("REJUDGED_JSON " + json.dumps(REJUDGED, ensure_ascii=False))

In [ ]:
# 4. 片付け・zip（採点し直した結果は回答の本文を含むので、リポジトリには点の要約だけを入れる）
kill(JUDGE_PROC)
(OUTP / "judge_s.json").write_text(json.dumps({"judge": JUDGE, "rejudged": REJUDGED}, ensure_ascii=False, indent=2) + "\n")
!cd /content && zip -qr judge_s.zip judge_s && ls -la /content/judge_s.zip